# AQME descriptors and conformer searches: notebook workflow

This notebook mirrors the command-line workshop in `README.md` using molecule **A1**. It uses native Python for RDKit, the AQME Python API for QDESCP, and a Python subprocess for CREST because CREST is an external executable.

Notebook-generated files are kept under `outputs/notebook/`, separate from command-line results.

## One-time Jupyter setup

The repository's `environment-aqme.yml` includes Jupyter. If you use an older/pre-existing `aqme` environment where Jupyter is not installed, run this once in a terminal:

```bash
conda install -n aqme -c conda-forge jupyterlab ipykernel
conda activate aqme
cd aqme_descriptor_workshop
jupyter lab AQME_descriptor_and_conformer_workshop.ipynb
```

Select the **aqme** Python kernel. Start Jupyter from this workshop directory so the path checks below succeed.

In [ ]:
# A line beginning with # is an explanatory comment. Python ignores it.
molecule_name = 'A1'  # A variable gives a value a reusable name.
print('Our example molecule is', molecule_name)

The previous cell contains a **comment**, a **variable**, a text value in quotation marks, and a **function call**. Run it with Shift-Return. Change `A1` to another name, run it again, and observe that the output changes.

In [ ]:
# A list stores several values in order.
mapped_atom_labels = [1, 2, 3]
print('Mapped atom labels:', mapped_atom_labels)
print('Number of mapped atoms:', len(mapped_atom_labels))

## Load the tools

Python begins with a small built-in vocabulary. An `import` loads additional tools. The next cell imports file-handling tools, pandas for tables, and RDKit for chemistry. You do not need to memorize the import lines.

In [ ]:
from importlib.metadata import version
from pathlib import Path
import csv
import os
import shutil
import subprocess
import sys

import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem, Draw

WORKSHOP_DIR = Path.cwd().resolve()
INPUT_CSV = WORKSHOP_DIR / 'inputs' / 'A1.csv'
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        'Start Jupyter from aqme_descriptor_workshop so inputs/A1.csv is available.'
    )
NOTEBOOK_OUTPUTS = WORKSHOP_DIR / 'outputs' / 'notebook'
NOTEBOOK_OUTPUTS.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('AQME:', version('aqme'))
print('RDKit:', rdBase.rdkitVersion)
print('CREST executable:', shutil.which('crest'))
print('Notebook outputs:', NOTEBOOK_OUTPUTS)

## 1. Read and inspect the mapped SMILES

The command line passes the CSV directly to AQME. In a notebook, we can inspect the same input before running anything.

In [ ]:
input_table = pd.read_csv(INPUT_CSV)
display(input_table)

code_name = input_table.loc[0, 'code_name']
smiles = input_table.loc[0, 'SMILES']
mol = Chem.MolFromSmiles(smiles)
if mol is None:
    raise ValueError(f'RDKit could not parse {smiles}')

formal_charge = Chem.GetFormalCharge(mol)
mapped_atoms = {
    atom.GetAtomMapNum(): atom.GetSymbol()
    for atom in mol.GetAtoms()
    if atom.GetAtomMapNum()
}
print('Formal charge:', formal_charge)
print('Mapped atoms:', mapped_atoms)
Draw.MolToImage(mol, size=(700, 250))

## 2. Convert SMILES to XYZ with native RDKit

Command-line equivalent: `python scripts/smiles_to_xyz.py`. The cell below exposes the same parse → add H → embed → optimize → write sequence.

In [ ]:
mol_3d = Chem.AddHs(Chem.Mol(mol))
embed_parameters = AllChem.ETKDGv3()
embed_parameters.randomSeed = 20260726
embed_parameters.enforceChirality = True
if AllChem.EmbedMolecule(mol_3d, embed_parameters) != 0:
    raise RuntimeError('ETKDGv3 embedding failed')

if AllChem.MMFFHasAllMoleculeParams(mol_3d):
    force_field = 'MMFF94'
    not_converged = AllChem.MMFFOptimizeMolecule(mol_3d, maxIters=1000)
else:
    force_field = 'UFF'
    not_converged = AllChem.UFFOptimizeMolecule(mol_3d, maxIters=1000)

xyz_dir = NOTEBOOK_OUTPUTS / '02_smiles_to_xyz'
xyz_dir.mkdir(parents=True, exist_ok=True)
xyz_path = xyz_dir / 'A1.xyz'
xyz_lines = Chem.MolToXYZBlock(mol_3d).splitlines()
xyz_lines[1] = (
    f'{code_name}; RDKit ETKDGv3/{force_field}; charge={formal_charge}; '
    'multiplicity=1; atom_maps=1:C,2:N,3:C'
)
xyz_path.write_text('\n'.join(xyz_lines) + '\n', encoding='utf-8')

print('Optimization converged:', not_converged == 0)
print('Wrote:', xyz_path)
print('\n'.join(xyz_lines[:8]))

## 3. Run an RDKit conformer search in Python

Command-line equivalent: `python scripts/rdkit_conformer_search.py`. We reuse its tested optimization and RMSD-pruning helpers, while keeping the ensemble and table live in the notebook.

In [ ]:
sys.path.insert(0, str(WORKSHOP_DIR))
from scripts.rdkit_conformer_search import (
    one_conformer_record,
    optimize_conformers,
    prune_by_heavy_atom_rmsd,
    xyz_block,
)

NUM_CONFS = 100
RMSD_THRESHOLD = 0.50
MAX_KEEP = 20

ensemble_mol = Chem.AddHs(Chem.Mol(mol))
parameters = AllChem.ETKDGv3()
parameters.randomSeed = 20260726
parameters.enforceChirality = True
parameters.pruneRmsThresh = -1.0
parameters.numThreads = 0
conformer_ids = list(
    AllChem.EmbedMultipleConfs(ensemble_mol, numConfs=NUM_CONFS, params=parameters)
)
method, optimization_results = optimize_conformers(ensemble_mol, conformer_ids)
ranked_ids = sorted(conformer_ids, key=lambda cid: optimization_results[cid][1])
kept_ids = prune_by_heavy_atom_rmsd(
    ensemble_mol, ranked_ids, RMSD_THRESHOLD, MAX_KEEP
)
minimum_energy = optimization_results[kept_ids[0]][1]

rdkit_rows = []
for rank, conformer_id in enumerate(kept_ids, start=1):
    not_converged, energy = optimization_results[conformer_id]
    rdkit_rows.append({
        'rank': rank,
        'conformer_id': conformer_id,
        'force_field': method,
        'energy_kcal_mol': energy,
        'relative_energy_kcal_mol': energy - minimum_energy,
        'optimization_converged': not_converged == 0,
    })
rdkit_summary = pd.DataFrame(rdkit_rows)

rdkit_dir = NOTEBOOK_OUTPUTS / '03_rdkit_search'
rdkit_dir.mkdir(parents=True, exist_ok=True)
rdkit_summary.to_csv(rdkit_dir / 'rdkit_summary.csv', index=False)
sdf_writer = Chem.SDWriter(str(rdkit_dir / 'A1_rdkit.sdf'))
ensemble_xyz = []
for row, conformer_id in zip(rdkit_rows, kept_ids):
    record = one_conformer_record(ensemble_mol, conformer_id)
    record.SetProp('_Name', f"A1_rdkit_conf_{row['rank']}")
    record.SetDoubleProp('energy_kcal_mol', row['energy_kcal_mol'])
    record.SetDoubleProp(
        'relative_energy_kcal_mol', row['relative_energy_kcal_mol']
    )
    sdf_writer.write(record)
    comment = (
        f"A1_rdkit_conf_{row['rank']}; {method}; "
        f"relative_energy_kcal_mol={row['relative_energy_kcal_mol']:.6f}; "
        f'charge={formal_charge}'
    )
    ensemble_xyz.append(xyz_block(record, comment))
sdf_writer.close()
(rdkit_dir / 'A1_rdkit_ensemble.xyz').write_text(
    ''.join(ensemble_xyz), encoding='utf-8'
)

print(f'Embedded {len(conformer_ids)} and retained {len(kept_ids)} conformers.')
display(rdkit_summary.round({
    'energy_kcal_mol': 4,
    'relative_energy_kcal_mol': 4,
}))

## 4. Generate descriptors through the AQME Python API

Command-line equivalent:

```bash
python -m aqme --qdescp --input inputs/A1.csv --qdescp_atoms "['1','2','3']"
```

The Python API runs the same AQME workflow. The working-directory context keeps AQME's log files with the notebook results.

In [ ]:
from aqme.qdescp import qdescp

RUN_QDESCP = True
qdescp_dir = NOTEBOOK_OUTPUTS / '04_aqme_descriptors'
qdescp_dir.mkdir(parents=True, exist_ok=True)

if RUN_QDESCP:
    original_directory = Path.cwd()
    try:
        os.chdir(qdescp_dir)
        qdescp(
            input=str(INPUT_CSV),
            qdescp_atoms=['1', '2', '3'],
            destination=str(qdescp_dir),
            nprocs=2,
        )
    finally:
        os.chdir(original_directory)
else:
    print('QDESCP skipped. Set RUN_QDESCP = True to run it.')

In [ ]:
interpret_path = qdescp_dir / 'AQME-ROBERT_interpret_A1.csv'
if interpret_path.exists():
    descriptor_table = pd.read_csv(interpret_path)
    preferred_columns = [
        'code_name', 'HOMO-LUMO gap', 'HOMO', 'LUMO', 'Dipole module',
        'Atom_1_C_Partial charge', 'Atom_2_N_Partial charge',
        'Atom_3_C_Partial charge',
    ]
    display(descriptor_table[[
        column for column in preferred_columns if column in descriptor_table.columns
    ]])
    print(f'{len(descriptor_table.columns)} total columns in {interpret_path.name}')
else:
    print('No descriptor table yet; run the preceding QDESCP cell.')

## 5. Run CREST from the notebook

CREST has no Python conformer-search API, so both terminal and notebook workflows ultimately launch the same executable. The notebook advantage is that the command, log, and subsequent analysis remain together.

A quick search took about 2.5 minutes with two threads during validation. Set `RUN_CREST = True` when ready.

In [ ]:
RUN_CREST = False
CREST_THREADS = 4
crest_dir = NOTEBOOK_OUTPUTS / '05_crest_quick'
crest_dir.mkdir(parents=True, exist_ok=True)
crest_input = crest_dir / 'A1.xyz'
crest_ensemble = crest_dir / 'crest_conformers.xyz'

if crest_ensemble.exists():
    print('Reusing existing ensemble:', crest_ensemble)
elif RUN_CREST:
    if shutil.which('crest') is None:
        raise RuntimeError('crest is not on PATH; select the aqme kernel.')
    shutil.copy2(xyz_path, crest_input)
    command = [
        'crest', 'A1.xyz', '--chrg', '-1', '--uhf', '0',
        '--gfn2', '--T', str(CREST_THREADS), '--quick',
    ]
    print('Running:', ' '.join(command))
    with (crest_dir / 'crest.log').open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command, cwd=crest_dir, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line)
        return_code = process.wait()
    if return_code != 0 or not crest_ensemble.exists():
        raise RuntimeError('CREST failed; inspect outputs/notebook/05_crest_quick/crest.log')
else:
    print('CREST skipped. Set RUN_CREST = True, or run the CLI version first.')

## 6. Compare notebook and command-line ensembles

The cell prefers the notebook CREST result, but it can reuse `outputs/04_crest_quick/crest_conformers.xyz` from the command-line workshop. Only relative energies are compared because MMFF and GFN2-xTB absolute energies are not on the same scale.

In [ ]:
from scripts.compare_ensembles import read_crest_xyz

crest_candidates = [
    crest_ensemble,
    WORKSHOP_DIR / 'outputs' / '04_crest_quick' / 'crest_conformers.xyz',
]
available_crest = next((path for path in crest_candidates if path.exists()), None)

if available_crest is None:
    print('No CREST ensemble found. Run the CREST cell or command-line step 4.')
else:
    crest_relative = read_crest_xyz(available_crest)
    comparison = pd.concat([
        rdkit_summary[['rank', 'relative_energy_kcal_mol']].assign(
            method='RDKit ETKDGv3/MMFF'
        ).rename(columns={'rank': 'conformer'}),
        pd.DataFrame({
            'conformer': range(1, len(crest_relative) + 1),
            'relative_energy_kcal_mol': sorted(crest_relative),
            'method': 'CREST/GFN2-xTB',
        }),
    ], ignore_index=True)
    display(comparison)
    display(
        comparison.groupby('method')['relative_energy_kcal_mol']
        .agg(['count', 'median', 'max']).round(3)
    )
    comparison.to_csv(
        NOTEBOOK_OUTPUTS / '06_relative_energy_comparison.csv', index=False
    )
    print('CREST source:', available_crest)

## Command line or notebook?

| Consideration | Command line | Notebook |
| --- | --- | --- |
| Best use | Repeatable batch runs and automation | Teaching, inspection, and exploratory analysis |
| Provenance | Shell history plus AQME/CREST logs | Code, narrative, tables, and outputs together |
| Scaling to many molecules | Strong | Easy to make slow or stateful |
| Debugging one molecule | More file inspection | Immediate variables and intermediate structures |
| RDKit execution | Script | Native Python cells |
| AQME execution | `python -m aqme` | AQME Python API |
| CREST execution | External executable | Same executable through `subprocess` |

**Recommended practice:** develop and explain the workflow in the notebook, then move stable parameters to the command-line workflow for production datasets. Keep input SMILES, charge, multiplicity, software versions, random seeds, and conformer filters identical when comparing the interfaces.